<a href="https://colab.research.google.com/github/Amritpal986/-Amritpal986.github.io/blob/main/AI_Powered_Student_Performance_Prediction_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# STUDENT PERFORMANCE PREDICTION SYSTEM
# Excel + Python + Machine Learning
# ============================================

# Install dependency
!pip install openpyxl -q

# Import Libraries
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# -----------------------------------
# Upload Excel File
# -----------------------------------

print("Upload student_data.xlsx")

uploaded = files.upload()

# Add a check to verify if a file was uploaded
if not uploaded:
    print("No file was uploaded. Please upload 'student_data.xlsx' to continue.")
    # Exit or handle the error appropriately, for now, we'll just print and stop execution for this section
    # In a real notebook, you might want to raise an exception or provide more interactive guidance.
    raise FileNotFoundError("No file uploaded.")

file_name = list(uploaded.keys())[0]

# -----------------------------------
# Load Dataset
# -----------------------------------

df = pd.read_excel(file_name)

print("\nDataset Preview:")
print(df.head())

# -----------------------------------
# Features and Target
# -----------------------------------

X = df[
    [
        "Students",
        "Attendance",
        "Research_Publications"
    ]
]

y = df["Placement_Percentage"]

# -----------------------------------
# Train-Test Split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------------
# Train Model
# -----------------------------------

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------------
# Prediction
# -----------------------------------

predictions = model.predict(X_test)

# -----------------------------------
# Accuracy
# -----------------------------------

mae = mean_absolute_error(y_test, predictions)

print("\nModel Trained Successfully")
print("Mean Absolute Error:", round(mae, 2))

# -----------------------------------
# Custom Prediction
# -----------------------------------

print("\n----- Student Prediction -----")

# Updated input prompts to match the new features
students = float(input("Number of Students: "))
attendance = float(input("Attendance (%): "))
research_publications = float(input("Research Publications: "))

sample = pd.DataFrame({
    "Students":[students],
    "Attendance":[attendance],
    "Research_Publications":[research_publications]
})

predicted_score = model.predict(sample)[0]

print("\nPredicted Final Score:", round(predicted_score,2))

# -----------------------------------
# Grade Category
# -----------------------------------

if predicted_score >= 85:
    grade = "Excellent"

elif predicted_score >= 70:
    grade = "Good"

elif predicted_score >= 50:
    grade = "Average"

else:
    grade = "Poor"

print("Performance Category:", grade)

## Data Visualization: Attendance vs. Placement Percentage

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Attendance', y='Placement_Percentage', data=df, hue='Department', s=100)
plt.title('Attendance vs. Placement Percentage by Department')
plt.xlabel('Attendance (%)')
plt.ylabel('Placement Percentage (%)')
plt.grid(True)
plt.show()

In [ ]:
print("Checking Dataset Quality...")

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
print("\nDataset Statistics")
print(df.describe())

In [ ]:
print("\nCorrelation Matrix")
print(df.corr(numeric_only=True))

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")
print(importance)

In [ ]:
df["Rank"] = df["Placement_Percentage"].rank(
    ascending=False,
    method="dense"
)

print(df[
    ["Placement_Percentage","Rank"]
].sort_values("Rank"))

In [ ]:
def grade(score):

    if score >= 90:
        return "A+"

    elif score >= 80:
        return "A"

    elif score >= 70:
        return "B"

    elif score >= 60:
        return "C"

    else:
        return "D"

df["Grade"] = df["Placement_Percentage"].apply(grade)

In [ ]:
def category(score):

    if score >= 85:
        return "Excellent"

    elif score >= 70:
        return "Good"

    elif score >= 50:
        return "Average"

    else:
        return "Poor"

df["Category"] = df["Placement_Percentage"].apply(category)

In [ ]:
output_file = "Student_Performance_Report.xlsx"

df.to_excel(
    output_file,
    index=False
)

print(
    f"\nReport saved as {output_file}"
)

In [ ]:
from google.colab import files

files.download(
    "Student_Performance_Report.xlsx"
)

In [ ]:
predicted_score = model.predict(sample)[0]

confidence = 100 - mae

print(
    f"Prediction Confidence: {confidence:.2f}%"
)

In [ ]:
print("\nPrediction for X_test[0:1]:", model.predict(X_test[0:1]))

In [ ]:
print("\nTop 5 Students")

top_students = df.sort_values(
    by="Placement_Percentage",
    ascending=False
)

print(top_students.head())

In [ ]:
risk_students = df[
    df["Placement_Percentage"] < 60
]

print("\nAt Risk Students")

print(risk_students)

In [ ]:
avg_score = df["Placement_Percentage"].mean()

highest = df["Placement_Percentage"].max()

lowest = df["Placement_Percentage"].min()

print("\nClass Summary")

print("Average Score:", avg_score)
print("Highest Score:", highest)
print("Lowest Score:", lowest)

In [ ]:
if attendance < 75:
    print(
        "Suggestion: Improve Attendance"
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for plotting: melt the DataFrame to have 'variable' and 'value' columns
df_melted = df.melt(id_vars=['Department'], value_vars=['Attendance', 'Placement_Percentage'], var_name='Metric', value_name='Value')

plt.figure(figsize=(12, 7))
sns.barplot(x='Department', y='Value', hue='Metric', data=df_melted, palette='viridis')
plt.title('Comparison of Attendance and Placement Percentage by Department')
plt.xlabel('Department')
plt.ylabel('Percentage')
plt.ylim(0, 100) # Set y-axis limits from 0 to 100 for percentages
plt.legend(title='Metric')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(
    model,
    "student_model.pkl"
)

print("Model Saved Successfully")